In [1]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)


Token loaded: True


In [2]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git


Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 114 (delta 49), reused 73 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 907.07 KiB | 18.90 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [3]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [4]:
!git config --global credential.helper store


In [5]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")


GitHub authentication configured.


In [6]:
!git fetch origin
!git switch nehna


branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [7]:
!git pull


Already up to date.


In [8]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA available: True
GPU: Tesla T4


In [9]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.1 MB/s eta 0:00:00


In [10]:
import json
import os
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device=device
)

print("Model loaded successfully.")
print("Device:", device)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded successfully.
Device: cuda


In [12]:
with open("data/corpus_ml.json", "r", encoding="utf-8") as f:
    corpus_ml = json.load(f)

with open("data/questions_ml.json", "r", encoding="utf-8") as f:
    questions_ml = json.load(f)

print("Malayalam corpus:", len(corpus_ml))
print("Malayalam questions:", len(questions_ml))

Malayalam corpus: 247
Malayalam questions: 100


In [13]:
corpus_texts_ml = [doc["text"] for doc in corpus_ml]
question_texts_ml = [q["question"] for q in questions_ml]

print("Number of passages:", len(corpus_texts_ml))
print("Number of questions:", len(question_texts_ml))

Number of passages: 247
Number of questions: 100


In [14]:
def embed_passages_ml(texts):
    prefixed = ["passage: " + t for t in texts]

    return model.encode(
        prefixed,
        normalize_embeddings=True,
        show_progress_bar=True
    )


passage_embeddings_ml = embed_passages_ml(corpus_texts_ml)

print("Passage embeddings shape:", passage_embeddings_ml.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Passage embeddings shape: (247, 768)


In [15]:
def embed_queries_ml(texts):
    prefixed = ["query: " + t for t in texts]

    return model.encode(
        prefixed,
        normalize_embeddings=True,
        show_progress_bar=True
    )


query_embeddings_ml = embed_queries_ml(question_texts_ml)

print("Query embeddings shape:", query_embeddings_ml.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Query embeddings shape: (100, 768)


In [16]:
print("Passage embedding dimension:", passage_embeddings_ml.shape[1])
print("Query embedding dimension:", query_embeddings_ml.shape[1])

assert passage_embeddings_ml.shape[1] == query_embeddings_ml.shape[1]

print("✓ Embedding dimensions match.")

Passage embedding dimension: 768
Query embedding dimension: 768
✓ Embedding dimensions match.


In [17]:
def dense_search_numpy_ml(query_vec, k=5):
    scores = passage_embeddings_ml @ query_vec

    ranked_ids = np.argsort(-scores)

    return ranked_ids[:k].tolist()

In [18]:
numpy_ids = dense_search_numpy_ml(
    query_embeddings_ml[0],
    k=5
)

print("Top-5 NumPy passage IDs:")
print(numpy_ids)

Top-5 NumPy passage IDs:
[49, 232, 190, 231, 134]


In [19]:
dim = passage_embeddings_ml.shape[1]

index_ml = faiss.IndexFlatIP(dim)

index_ml.add(passage_embeddings_ml)

print("FAISS index created.")
print("Number of indexed passages:", index_ml.ntotal)

FAISS index created.
Number of indexed passages: 247


In [20]:
def dense_search_faiss_ml(query_vec, k=5):
    query_vec = query_vec.reshape(1, -1)

    scores, ids = index_ml.search(query_vec, k)

    return ids[0].tolist()

In [21]:
faiss_ids = dense_search_faiss_ml(
    query_embeddings_ml[0],
    k=5
)

numpy_ids = dense_search_numpy_ml(
    query_embeddings_ml[0],
    k=5
)

print("FAISS:", faiss_ids)
print("NumPy:", numpy_ids)

assert faiss_ids == numpy_ids, \
    "FAISS and NumPy results do not match!"

print("✓ FAISS and NumPy results match exactly.")

FAISS: [49, 232, 190, 231, 134]
NumPy: [49, 232, 190, 231, 134]
✓ FAISS and NumPy results match exactly.


In [22]:
dense_results_ml = {}

for i, q in enumerate(questions_ml):
    dense_results_ml[q["question_id"]] = dense_search_faiss_ml(
        query_embeddings_ml[i],
        k=10
    )

print("Questions processed:", len(dense_results_ml))

Questions processed: 100


In [23]:
def recall_at_k(questions, results, k):
    hits = 0

    for q in questions:
        gold_id = q["gold_passage_id"]
        retrieved = results[q["question_id"]][:k]

        if gold_id in retrieved:
            hits += 1

    return hits / len(questions)


def mrr(questions, results):
    reciprocal_ranks = []

    for q in questions:
        gold_id = q["gold_passage_id"]
        retrieved = results[q["question_id"]]

        if gold_id in retrieved:
            rank = retrieved.index(gold_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    return sum(reciprocal_ranks) / len(questions)

In [24]:
for k in [1, 3, 5, 10]:
    print(
        f"Malayalam Dense Recall@{k}:",
        recall_at_k(
            questions_ml,
            dense_results_ml,
            k
        )
    )

print(
    "Malayalam Dense MRR:",
    mrr(
        questions_ml,
        dense_results_ml
    )
)

Malayalam Dense Recall@1: 0.66
Malayalam Dense Recall@3: 0.84
Malayalam Dense Recall@5: 0.9
Malayalam Dense Recall@10: 0.93
Malayalam Dense MRR: 0.7566666666666667


In [25]:
os.makedirs("results", exist_ok=True)

with open(
    "results/dense_top10_ml.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        dense_results_ml,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved: results/dense_top10_ml.json")

Saved: results/dense_top10_ml.json


In [26]:
with open(
    "results/dense_top10_ml.json",
    "r",
    encoding="utf-8"
) as f:
    dense_results_check_ml = json.load(f)

print("Loaded saved results:", len(dense_results_check_ml))

assert len(dense_results_check_ml) == 100

print("✓ Dense retrieval results saved and verified.")

Loaded saved results: 100
✓ Dense retrieval results saved and verified.


## Malayalam Dense Retrieval Results

| Metric | Score |
|---|---:|
| Recall@1 | 0.66 |
| Recall@3 | 0.84 |
| Recall@5 | 0.9 |
| Recall@10 | 0.93 |
| MRR | 0.757 |

Dense retrieval was evaluated on the same 100 Malayalam questions used for the BM25 baseline.

## BM25 vs Dense Retrieval — Malayalam

| Metric | BM25 | Dense |
|---|---:|---:|
| Recall@1 | 0.65 | 0.66 |
| Recall@3 | 0.75 | 0.84 |
| Recall@5 | 0.82 | 0.9 |
| Recall@10 | 0.89 | 0.93 |
| MRR | 0.7152 | 0.757 |

Both systems were evaluated on the same 100 Malayalam questions.